# CPS damping planner

Coupling-Phase Spectroscopy — governed Colab runner.

Consumes a reduced operator produced by a prior notebook and ranks a preregistered damping grid. The recommendation is a continuation-run hypothesis; it does not alter model weights in this notebook.

In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[pythia,notebooks]"], check=True)
print("repo", repo, "ref", GIT_REF)

In [ ]:
import glob, numpy as np, pathlib
from cps.pythia.planner import damping_family, plan_scalar_control
paths=glob.glob("/content/cps-artifacts/**/reduced_operator.npy", recursive=True)
if not paths:
    raise FileNotFoundError("Run a probe notebook first or copy a reduced_operator.npy into /content/cps-artifacts")
path=pathlib.Path(paths[-1]); A=np.load(path)
edges=[tuple(x) for x in np.argwhere(np.abs(A-np.diag(np.diag(A)))>0)[:8]]
recommendation=plan_scalar_control("isotropic_damping",0.0,[0.0,0.02,0.05,0.1,0.2],lambda g:damping_family(A,g),edges)
print(recommendation)

In [ ]:
import pathlib, shutil
export_dir = pathlib.Path("/content/cps-export")
export_dir.mkdir(parents=True, exist_ok=True)
source = pathlib.Path("/content/cps-artifacts")
if source.exists():
    shutil.copytree(source, export_dir / "artifacts", dirs_exist_ok=True)
shutil.make_archive("/content/cps-export", "zip", "/content/cps-export")
print("exported", export_dir)